<img src="https://github.com/nicholasmetherall/digital-earth-pacific-macblue-activities/blob/main/attachments/images/DE_Pacific_banner.JPG?raw=true" width="900"/>

Figure 1.1.a. Jupyter environment + Python notebooks

# Digital Earth Pacific Notebook 1 prepare postcard and load data to csv

The objective of this notebook is to prepare a geomad postcard for your AOI (masking, scaling and loading additional band ratios and spectral indices) and sampling all the datasets into a csv based on your training data geodataframe.

## Step 1.1: Configure the environment

In [1]:
from datetime import datetime
from shapely.geometry import Polygon
from shapely import box
from pyproj import CRS 
import folium
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio as rio
import xarray as xr
import rioxarray
import joblib
from ipyleaflet import basemaps
import pystac_client
from dask.distributed import Client as DaskClient
from odc.stac import load, configure_s3_access
import planetary_computer
from odc.stac import load
from pystac.client import Client
from skimage.feature import graycomatrix, graycoprops
import matplotlib.pyplot as plt
from matplotlib import colors
from utils import load_data, load_s1_dem, scale, calculate_band_indices, apply_mask, mask_water, all_masks, do_prediction, process_year

In [2]:
# Reload scripts and imports
%load_ext autoreload
%autoreload 2

In [3]:
# Predefined variable for title and version

# Enter your initials
initials = "nm"

# Enter your site name
site = "tongatapu"

# Date
date = datetime.now()

# Make a clean version string
version = f"{initials}-{site}-{date.strftime('%d%m%Y')}"
print(version)

nm-tongatapu-06072026


## Step 1.2: Configure STAC access and search parameters

In [4]:
catalog = "https://stac.digitalearthpacific.org"
client = Client.open(catalog)

In [5]:
## Use training data bounds

# training = gpd.read_file(f"training-data/nm-{site}.geojson")
training = gpd.read_file("training-data/nm-tongatapu-11122025_postcard_4.geojson")
training = training.to_crs("EPSG:4326")
min_lon, min_lat, max_lon, max_lat = training.total_bounds

bbox = [min_lon, min_lat, max_lon, max_lat]

In [6]:
training

,LULC_code,LULC_class,geometry
0,6,Water,POINT (-175.32648 -21.05595)
1,6,Water,POINT (-175.36156 -21.09345)
2,6,Water,POINT (-175.12939 -21.27945)
3,6,Water,POINT (-175.02447 -21.1477)
4,6,Water,POINT (-175.31365 -21.16867)
...,...,...,...
985,4,Settlements,POINT (-175.1994 -21.13518)
986,4,Settlements,POINT (-175.19913 -21.13671)
987,4,Settlements,POINT (-175.20174 -21.13612)
988,4,Settlements,POINT (-175.20039 -21.13224)


In [7]:
training.columns.unique()

Index(['LULC_code', 'LULC_class', 'geometry'], dtype='object')

In [8]:
# bbox = [178.410921, -18.188382, 178.46952, -18.14731]

In [9]:
model = joblib.load("models/nm-tongatapu-11122025-test.model")

In [ ]:
# In 4-time-series.ipynb or a new notebook
years = [2020, 2021, 2022, 2023, 2024, 2025]
classifications = {}
masks = {}

for year in years:
    predictions, mask = process_year(year, bbox, client, model, version)
    if predictions is not None:
        classifications[year] = predictions
        masks[year] = mask
        # predictions.rio.to_raster(f"outputs/lulc_{year}.tif")

# Now ready for change detection analysis

✓ Found 1 items for 2020
✓ Classification complete for 2020
✓ Found 1 items for 2021
✓ Classification complete for 2021
✓ Found 1 items for 2022
✓ Classification complete for 2022
✓ Found 1 items for 2023
✓ Classification complete for 2023
✓ Found 1 items for 2024
✓ Classification complete for 2024
✓ Found 1 items for 2025


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap



# Updated classes list to include No Data (Code 0)
lulc_classes = [
    [0, 'No_Data', '#00000000'], # <-- NEW: Code 0 mapped to a light gray color
    [1, 'Forest_land', '#064a00'],
    [2, 'Grazing_Cropland', '#FFEE8C' ],
    [3, 'Wetland', '#73ffd2'],
    [4, 'Settlements', '#bd0007'],
    [5, 'Bare_land','#919191'],
    [6, 'Water','#71a8ff'],
]

# # Define your LULC classes and colors
# lulc_classes = {
#     1: "Water",
#     2: "Vegetation",
#     3: "Agriculture", 
#     4: "Settlements",
#     5: "Bare Ground"
#     # Adjust based on your actual classes

from matplotlib import colors as mcolors # Safe import name

values_list = [c[0] for c in lulc_classes]
color_list = [c[2] for c in lulc_classes]  # Named distinctively

# Use the imported module to build the colormap from your list
cmap = mcolors.ListedColormap(color_list) 

# If you use the bounds/norm later, call them from the module too:
bounds = values_list + [values_list[-1] + 1]
norm = mcolors.BoundaryNorm(bounds, cmap.N)

# cmap = c_map

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, year in enumerate(sorted(classifications.keys())):
    ax = axes[idx]
    im = ax.imshow(classifications[year].values, cmap=cmap, vmin=1, vmax=len(lulc_classes))
    ax.set_title(f"LULC {year}", fontsize=14, fontweight='bold')
    ax.axis('off')

cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
cbar = fig.colorbar(im, cax=cbar_ax, ticks=range(1, len(lulc_classes)+1))
cbar.ax.set_yticklabels(list(lulc_classes.values()))
cbar.set_label('LULC Class', fontsize=12)

plt.suptitle('LULC Classification Time Series - Tongatapu', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/lulc_grid.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
df_change = pd.DataFrame(change_data).T.fillna(0)

# Add these temporary debug lines:
print("DataFrame Shape (Rows, Columns):", df_change.shape)
print("Current Columns:", list(df_change.columns))
print("LULC Classes Dictionary:", lulc_dict)

# Your renaming line:
df_change.columns = [lulc_dict.get(int(col), f"Class {col}") for col in df_change.columns]

In [ ]:
import pandas as pd
import numpy as np

# Calculate pixel counts for each class per year
change_data = {}

for year, classification in classifications.items():
    unique, counts = np.unique(classification.values, return_counts=True)
    change_data[year] = dict(zip(unique, counts))

# 1. Create your dictionary mapping
lulc_dict = {i: name for i, name in enumerate(lulc_classes)}

# 2. Map the columns ONCE (Make sure to remove the duplicate line below it)
df_change.columns = [lulc_dict.get(int(col), f"Class {col}") for col in df_change.columns]

# Convert pixel counts to area (e.g., if 10m resolution, multiply by 100 m²)
pixel_size_m2 = 100  # 10m x 10m pixels
df_area = df_change * pixel_size_m2 / 10000  # Convert to hectares
df_area.index.name = 'Year'

# Plot 1: Stacked area chart
fig, ax = plt.subplots(figsize=(12, 6))
df_area.plot(kind='area', stacked=True, ax=ax, alpha=0.7, color=colors[:len(df_area.columns)])
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Area (hectares)', fontsize=12)
ax.set_title('LULC Change Detection - Tongatapu', fontsize=14, fontweight='bold')
ax.legend(title='LULC Class', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/lulc_area_change.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

for col in df_area.columns:
    ax.plot(df_area.index, df_area[col], marker='o', linewidth=2.5, markersize=8, label=col)

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Area (hectares)', fontsize=12)
ax.set_title('LULC Class Evolution - Tongatapu', fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='best')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/lulc_trends.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Calculate year-to-year absolute changes (hectares)
df_changes = df_area.diff().fillna(0)

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(df_changes.T, cmap='RdBu_r', aspect='auto', interpolation='nearest')

ax.set_xticks(range(len(df_changes)))
ax.set_xticklabels(df_changes.index)
ax.set_yticks(range(len(df_changes.columns)))
ax.set_yticklabels(df_changes.columns)
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('LULC Class', fontsize=12)
ax.set_title('Year-to-Year LULC Change (hectares)', fontsize=14, fontweight='bold')

cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Change (ha)', fontsize=11)

# Add values in cells
for i in range(len(df_changes)):
    for j in range(len(df_changes.columns)):
        text = ax.text(i, j, f'{df_changes.iloc[i, j]:.0f}',
                      ha="center", va="center", color="black", fontsize=9)

plt.tight_layout()
plt.savefig('outputs/lulc_change_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
def calculate_transition_matrix(classification_t0, classification_t1, lulc_classes):
    """Calculate transitions between two years"""
    n_classes = len(lulc_classes)
    transition = np.zeros((n_classes, n_classes))
    
    # Flatten arrays
    t0_flat = classification_t0.values.flatten()
    t1_flat = classification_t1.values.flatten()
    
    # Valid pixels (not masked)
    valid = ~(np.isnan(t0_flat) | np.isnan(t1_flat))
    t0_flat = t0_flat[valid].astype(int)
    t1_flat = t1_flat[valid].astype(int)
    
    # Build transition matrix
    for i in range(len(t0_flat)):
        from_class = t0_flat[i] - 1
        to_class = t1_flat[i] - 1
        if 0 <= from_class < n_classes and 0 <= to_class < n_classes:
            transition[from_class, to_class] += 1
    
    return transition

# Calculate transition from 2023 to 2024 (example)
transition_matrix = calculate_transition_matrix(
    classifications[2023], 
    classifications[2024], 
    lulc_classes
)

# Plot transition matrix
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(transition_matrix, cmap='YlOrRd', interpolation='nearest')

class_names = list(lulc_classes.values())
ax.set_xticks(range(len(class_names)))
ax.set_yticks(range(len(class_names)))
ax.set_xticklabels(class_names, rotation=45, ha='right')
ax.set_yticklabels(class_names)
ax.set_xlabel('To (2024)', fontsize=12, fontweight='bold')
ax.set_ylabel('From (2023)', fontsize=12, fontweight='bold')
ax.set_title('LULC Transition Matrix 2023→2024', fontsize=14, fontweight='bold')

# Add pixel counts
for i in range(len(class_names)):
    for j in range(len(class_names)):
        text = ax.text(j, i, f'{int(transition_matrix[i, j])}',
                      ha="center", va="center", color="black", fontsize=10, fontweight='bold')

cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Pixel Count', fontsize=11)

plt.tight_layout()
plt.savefig('outputs/transition_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Summary statistics
summary_stats = pd.DataFrame({
    'Year': list(classifications.keys()),
    'Total Area (ha)': [df_area.loc[year].sum() for year in classifications.keys()],
    'Dominant Class': [df_area.loc[year].idxmax() for year in classifications.keys()],
    'Dominant % of Total': [(df_area.loc[year].max() / df_area.loc[year].sum() * 100) for year in classifications.keys()]
})

print(summary_stats.to_string(index=False))